<a href="https://colab.research.google.com/github/ambika-1513/Computer-vision-learning/blob/main/CNN_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

In [3]:
torch.manual_seed(42)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [5]:
df = pd.read_csv("/content/fashion-mnist_train.csv")
df

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59995,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59996,1,0,0,0,0,0,0,0,0,0,...,73,0,0,0,0,0,0,0,0,0
59997,8,0,0,0,0,0,0,0,0,0,...,160,162,163,135,94,0,0,0,0,0
59998,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [7]:
X = df.iloc[:,1:].values
y = df.iloc[:, 0].values

In [9]:
x_train, x_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state = 42)


In [10]:
x_train = x_train/255.0
x_test = x_test/255.0

In [19]:
class CustomDataset:
  def __init__(self, features, labels):
    self.features = torch.tensor(features , dtype=torch.float32).reshape(-1,1,28,28)
    self.labels = torch.tensor(labels, dtype = torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self, idx):
    return self.features[idx], self.labels[idx]

In [20]:
train_dataset = CustomDataset(x_train, y_train)
test_dataset = CustomDataset(x_test, y_test)

In [28]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory = True)

In [29]:
len(train_loader), len(test_loader)

(1500, 375)

In [35]:
class NeuralNetwork(nn.Module):
  def __init__(self, input_features):
    super().__init__()
    self.features = nn.Sequential(
        nn.Conv2d(input_features, 32, kernel_size=3, padding="same"),
        nn.ReLU(),
        nn.BatchNorm2d(32),
        nn.MaxPool2d(kernel_size=2, stride=2),
        nn.Conv2d(32, 64, kernel_size=3, padding="same"),
        nn.ReLU(),
        nn.BatchNorm2d(64), # Ensuring this is 64 to match the previous Conv2d output
        nn.MaxPool2d(kernel_size=2, stride=2)
        )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(64*7*7, 128),
        nn.ReLU(),
        nn.Dropout(p = 0.4),
        nn.Linear(128, 64),
        nn.ReLU(),
        nn.Dropout(p = 0.4),
        nn.Linear(64, 10)
    )
  def forward(self,x):
    x = self.features(x)
    x = self.classifier(x)
    return x

In [36]:
learning_rate= 0.01
epochs = 100

In [37]:
model = NeuralNetwork(1)
model = model.to(device)
optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay = 1e-4)
criterion = nn.CrossEntropyLoss()

In [38]:
for epoch in range(epochs):
  total_epoch_loss = 0
  for batch_features, batch_labels in train_loader:
    batch_features = batch_features.to(device).float()
    batch_labels = batch_labels.to(device)
    optimizer.zero_grad()
    outputs = model(batch_features)
    loss = criterion(outputs, batch_labels)
    loss.backward()
    optimizer.step()
    total_epoch_loss += loss.item()
  print(f"Epoch: {epoch+1}, Loss: {total_epoch_loss/len(train_loader)}")

Epoch: 1, Loss: 0.6419911663333575
Epoch: 2, Loss: 0.38230486390988033
Epoch: 3, Loss: 0.32838194304704665
Epoch: 4, Loss: 0.29272680916388827
Epoch: 5, Loss: 0.2657138318990668
Epoch: 6, Loss: 0.24582179387658834
Epoch: 7, Loss: 0.23160378645112117
Epoch: 8, Loss: 0.21912358616665006
Epoch: 9, Loss: 0.2009566865656525
Epoch: 10, Loss: 0.19028945855113366
Epoch: 11, Loss: 0.18164772320787112
Epoch: 12, Loss: 0.17132639643580963
Epoch: 13, Loss: 0.1574494223659858
Epoch: 14, Loss: 0.15208251997440433
Epoch: 15, Loss: 0.1414245010719945
Epoch: 16, Loss: 0.13777533145621418
Epoch: 17, Loss: 0.12814792433256905
Epoch: 18, Loss: 0.12389123965753242
Epoch: 19, Loss: 0.11469644686641793
Epoch: 20, Loss: 0.10917237897869199
Epoch: 21, Loss: 0.10731819634131777
Epoch: 22, Loss: 0.1036908744132767
Epoch: 23, Loss: 0.09808827894025793
Epoch: 24, Loss: 0.09233419750103106
Epoch: 25, Loss: 0.08758427234180272
Epoch: 26, Loss: 0.0826460030246526
Epoch: 27, Loss: 0.07879068587852331
Epoch: 28, Loss: 

In [39]:
model.eval()

NeuralNetwork(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (5): ReLU()
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.4, inplace=False)
    (7): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [40]:
total= 0
correct = 0
with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    batch_features = batch_features.to(device).float()
    batch_labels = batch_labels.to(device)
    outputs = model(batch_features)
    _, predicted = torch.max(outputs.data, 1)
    total += batch_labels.size(0)
    correct += (predicted == batch_labels).sum().item()
print(correct/total)

0.92275
